In [ ]:
import os
import cv2
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm

# Ayarlar
INFANT_DATA_PATH = r"C:\Users\Ayberk\Desktop\Vessel_Multiclass_Data"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 2  # EfficientNet-B5 yüksek VRAM tüketir, 2 güvenlidir
IMG_SIZE = (512, 512) 
LEARNING_RATE = 1e-5 # Fine-tuning için düşük hız (önceki bilgileri korumak için)
EPOCHS = 100

In [ ]:


class InfantFundusDataset(Dataset):
    def __init__(self, root_dir, split="train", transform=None):
        self.img_dir = os.path.join(root_dir, split, "image")
        self.mask_dir = os.path.join(root_dir, split, "mask")
        
        # Sadece resim dosyalarını al
        valid_extensions = ('.png', '.jpg', '.jpeg', '.tif', '.bmp')
        self.image_names = [f for f in os.listdir(self.img_dir) 
                            if f.lower().endswith(valid_extensions)]
        
        self.transform = transform
        self.clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        print(f"--> {split} seti için {len(self.image_names)} geçerli dosya bulundu.")

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        img_name = self.image_names[idx]
        img_path = os.path.join(self.img_dir, img_name)
        
        # Uzantıyı ayır (örn: "638743611915835469_fundus")
        base_name = os.path.splitext(img_name)[0]
        
        # MASKESİ VAR MI KONTROL ET (Farklı uzantıları dene)
        mask_path = None
        for ext in ['.png', '.jpg', '.jpeg', '.tif']:
            temp_path = os.path.join(self.mask_dir, base_name + ext)
            if os.path.exists(temp_path):
                mask_path = temp_path
                break

        image = cv2.imread(img_path)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE) if mask_path else None

        # Detaylı Hata Mesajı
        if image is None:
            raise FileNotFoundError(f"RESİM OKUNAMADI: {img_path}")
        if mask is None:
            raise FileNotFoundError(f"MASKE BULUNAMADI! Resim: {img_name} mevcut ama maske klasöründe '{base_name}' isminde uygun uzantılı bir dosya yok.")

        # --- CLAHE Önişleme (Damarları netleştirir) ---
        lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        l = self.clahe.apply(l)
        lab = cv2.merge((l, a, b))
        image = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']

        return image, mask

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import DataLoader

# 1. Görüntü Dönüştürme Ayarları (Transforms)
train_transform = A.Compose([
    A.PadIfNeeded(
        min_height=768, 
        min_width=768, 
        border_mode=cv2.BORDER_CONSTANT, 
        value=0
    ),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.Affine(
        translate_percent={"x": (-0.05, 0.05), "y": (-0.05, 0.05)}, 
        scale=(0.95, 1.05), 
        rotate=(-10, 10), 
        p=0.5
    ),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.PadIfNeeded(
        min_height=768, 
        min_width=768, 
        border_mode=cv2.BORDER_CONSTANT, 
        value=0
    ),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

# 2. Dataset Nesnelerini Oluşturma (Hatanın Çözümü Burada)
# InfantFundusDataset sınıfının Hücre 2'de tanımlanmış olması gerekir.
train_ds = InfantFundusDataset(INFANT_DATA_PATH, split="train", transform=train_transform)
val_ds = InfantFundusDataset(INFANT_DATA_PATH, split="val", transform=val_transform)

# 3. DataLoader Nesnelerini Oluşturma
# Eğer "CUDA out of memory" hatası alırsan batch_size=1 yapabilirsin.
train_loader = DataLoader(train_ds, batch_size=2, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=2, shuffle=False)

print(f"--> Yükleme tamamlandı: {len(train_ds)} eğitim, {len(val_ds)} doğrulama görüntüsü.")

In [ ]:
# 1. Modeli 3 sınıf (BG, Arter, Ven) için kur
model = smp.UnetPlusPlus(
    encoder_name="efficientnet-b5",
    encoder_weights=None, 
    in_channels=3,
    classes=3, 
).to(DEVICE)

# 2. Yetişkin modelinden ağırlıkları yükle
pretrained_path = "best_fundus_model_efficient.pth"
if os.path.exists(pretrained_path):
    checkpoint = torch.load(pretrained_path)
    model_dict = model.state_dict()
    
    # Sadece boyutu uyan katmanları al (Segmentation Head hariç her yer)
    filtered_dict = {k: v for k, v in checkpoint.items() if k in model_dict and v.size() == model_dict[k].size()}
    model_dict.update(filtered_dict)
    model.load_state_dict(model_dict)
    print(f"--> {len(filtered_dict)} katman başarıyla aktarıldı.")
else:
    print("!! Önceden eğitilmiş model bulunamadı, sıfırdan başlanıyor.")

# Tversky Loss: alpha=0.3 (Eksikliği cezalandır), beta=0.7 (Gereksiz kalınlığı/FP cezalandır)
# beta > alpha yaparak modelin damarları "şişirmesini" engelliyoruz.
tversky_loss = smp.losses.TverskyLoss(mode='multiclass', alpha=0.3, beta=0.7)
focal_loss = smp.losses.FocalLoss(mode='multiclass')

def criterion(y_pred, y_true):
    # İki kaybı birleştirerek hem hassasiyet hem de denge sağlıyoruz
    return tversky_loss(y_pred, y_true) + focal_loss(y_pred, y_true)

# Öğrenme oranını biraz daha düşürebiliriz (Fine-tuning hassasiyeti için)
optimizer = torch.optim.Adam(model.parameters(), lr=5e-6)

In [ ]:
best_iou = 0.0

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    for images, masks in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        images, masks = images.to(DEVICE), masks.to(DEVICE).long()
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    model.eval()
    tp, fp, fn, tn = 0, 0, 0, 0
    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(DEVICE), masks.to(DEVICE).long()
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)
            
            stats = smp.metrics.get_stats(preds, masks, mode='multiclass', num_classes=3)
            tp += stats[0]; fp += stats[1]; fn += stats[2]; tn += stats[3]

    current_iou = smp.metrics.iou_score(tp, fp, fn, tn, reduction="macro-imagewise").item()
    print(f"Train Loss: {train_loss/len(train_loader):.4f} | Val IoU: {current_iou:.4f}")

    if current_iou > best_iou:
        best_iou = current_iou
        torch.save(model.state_dict(), "best_bebek_infant_model3.pth")
        print("--> Bebek modeli kaydedildi!")

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
import segmentation_models_pytorch as smp

# 1. Renk Düzeltme (Denormalizasyon)
# Orijinal fundus renklerini geri getirir 
def denormalize(img_tensor):
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = img_tensor.permute(1, 2, 0).cpu().numpy()
    img = (img * std) + mean 
    return np.clip(img, 0, 1)

# 2. Bebek Datası Renk Haritası (0: Arka Plan, 1: Arter, 2: Ven) [cite: 86, 193]
infant_colors = {0: [0,0,0], 1: [255,0,0], 2: [0,0,255]}

def infant_mask_to_rgb(mask):
    h, w = mask.shape
    rgb = np.zeros((h, w, 3), dtype=np.uint8)
    for cls, color in infant_colors.items():
        rgb[mask == cls] = color
    return rgb

# 3. Model Yükleme
model.load_state_dict(torch.load("best_bebek_infant_model3.pth"))
model.eval()

print(f"--- Bebek Datası Analizi (İlk 10 Örnek) ---")

with torch.no_grad():
    for i in range(min(10, len(val_ds))):
        image, mask = val_ds[i]
        input_tensor = image.unsqueeze(0).to(DEVICE)
        
        output = model(input_tensor)
        pred = torch.argmax(output, dim=1).squeeze(0)
        
        # Metrik İstatistikleri [cite: 178]
        # num_classes=3 (BG, Arter, Ven)
        tp, fp, fn, tn = smp.metrics.get_stats(
            pred.unsqueeze(0), # Batch boyutu ekle: [1, H, W]
            mask.unsqueeze(0).to(DEVICE).long(), 
            mode='multiclass', 
            num_classes=3
        )
        
        # HATA DÜZELTMESİ: reduction="micro" kullanarak boyut hatasını önlüyoruz
        # Artery (Sınıf 1) ve Vein (Sınıf 2) Dice Skorları [cite: 188]
        artery_dice = smp.metrics.f1_score(tp[:, 1], fp[:, 1], fn[:, 1], tn[:, 1], reduction="micro").item()
        vein_dice = smp.metrics.f1_score(tp[:, 2], fp[:, 2], fn[:, 2], tn[:, 2], reduction="micro").item()

        # Panel Oluşturma
        plt.figure(figsize=(16, 5))
        
        # Orijinal (Normalizasyon geri alınmış )
        plt.subplot(1, 3, 1)
        plt.title(f"Bebek Fundus: {val_ds.image_names[i]}")
        plt.imshow(denormalize(image))
        plt.axis('off')

        # Gerçek Etiket [cite: 148]
        plt.subplot(1, 3, 2)
        plt.title("Uzman Etiketi (GT)")
        plt.imshow(infant_mask_to_rgb(mask.numpy() if torch.is_tensor(mask) else mask))
        plt.axis('off')

        # Model Tahmini ve Skorlar 
        plt.subplot(1, 3, 3)
        plt.title(f"Tahmin\nArter Dice: {artery_dice:.4f} | Ven Dice: {vein_dice:.4f}")
        plt.imshow(infant_mask_to_rgb(pred.cpu().numpy()))
        plt.axis('off')
        
        plt.tight_layout()
        plt.show()